# 🦀 Day 1: Lifetime Annotations

Today:
- Why lifetimes exist
- The borrow checker
- Lifetime annotation syntax `'a`
- Annotating functions that return references

---

## ⏳ Why Lifetimes?

Lifetimes tell the compiler **how long references are valid**.
They prevent **dangling references** — pointers to freed memory.

```rust
// The compiler needs to know: does the returned &str
// live as long as `a` or as long as `b`?
fn longest(a: &str, b: &str) -> &str  // ❌ Won't compile!
```

Without lifetime annotations, the compiler can't guarantee the returned reference is valid.

## 🏷️ Lifetime Annotation Syntax

In [ ]:
// 'a is a lifetime parameter — it says:
// "the returned reference lives at least as long as BOTH inputs"

fn longest<'a>(a: &'a str, b: &'a str) -> &'a str {
    if a.len() >= b.len() { a } else { b }
}

let result;
{
    let s1 = String::from("long string is long");
    let s2 = String::from("xyz");
    result = longest(&s1, &s2);
    println!("Longest: {}", result); // ✅ Both s1 and s2 are still alive
}

// This would fail:
// let result;
// {
//     let s1 = String::from("hello");
//     result = longest(&s1, "world");
// } // s1 dropped here
// println!("{}", result); // ❌ result might reference dropped s1

In [ ]:
// Lifetime means: the return value is valid for the SHORTER of the two input lifetimes

let s1 = String::from("hello world");
let result;
{
    let s2 = String::from("hi");
    result = longest(s1.as_str(), s2.as_str());
    println!("Inside: {}", result); // ✅ Both alive
} // s2 dropped — result would be invalid if it referenced s2

// If result = "hello world" (from s1), it's actually still valid,
// but the compiler is conservative — it assumes the worst case

## 🔍 When Do You Need Lifetime Annotations?

**You need annotations when returning a reference AND:**
1. The function takes multiple reference parameters
2. The compiler can't figure out which input the output borrows from

In [ ]:
// One reference param → no annotation needed (compiler infers)
fn first_char(s: &str) -> &str {
    &s[..1]
}
println!("First char: {}", first_char("hello"));

// Two reference params, returning one → need annotation
fn pick<'a>(a: &'a str, b: &'a str, first: bool) -> &'a str {
    if first { a } else { b }
}
println!("Pick: {}", pick("yes", "no", true));

In [ ]:
// Different lifetimes for different purposes
fn search<'a, 'b>(text: &'a str, _query: &'b str) -> &'a str {
    // Return always comes from 'text', not 'query'
    // So only 'a matters for the return
    if text.contains(_query) {
        text
    } else {
        ""
    }
}

let text = String::from("hello world");
let result;
{
    let query = String::from("hello");
    result = search(&text, &query);
} // query dropped — but result borrows from text, not query
println!("Found: '{}'", result); // ✅ text is still alive

## 📝 Lifetime Annotations Don't Change Lifetimes!

They don't make references live longer or shorter.
They just **describe** the relationships between lifetimes
so the compiler can check them.

In [ ]:
// Multiple return patterns

// Return borrows from first param only
fn first_of<'a>(a: &'a str, _b: &str) -> &'a str {
    a
}

// Return borrows from self (common in methods)
struct Document {
    content: String,
}

impl Document {
    fn title(&self) -> &str {  // Lifetime elided — borrows from &self
        let end = self.content.find('\n').unwrap_or(self.content.len());
        &self.content[..end]
    }
}

let doc = Document { content: "My Title\nBody text here".into() };
println!("Title: {}", doc.title());

let a = "hello";
let b = "world";
println!("First: {}", first_of(a, b));

## 🏋️ Exercises

In [ ]:
// Exercise 1: Add lifetime annotations to make this compile
// fn longest_with_announcement(x: &str, y: &str, ann: &str) -> &str {
//     println!("Announcement: {}", ann);
//     if x.len() > y.len() { x } else { y }
// }

// YOUR CODE HERE


In [ ]:
// Exercise 2: Write a function that returns the first word that's longer
// than a given length, with proper lifetime annotations
// fn first_long_word(text: &str, min_len: usize) -> Option<&str>

// YOUR CODE HERE


## 🎯 Key Takeaways

1. Lifetimes prevent **dangling references** at compile time
2. `'a` is a lifetime parameter — it says "lives at least this long"
3. Annotations describe **relationships**, they don't change actual lifetimes
4. The output lifetime is the **minimum** of the related input lifetimes
5. Single-parameter functions usually don't need annotations (elision)

---
**Tomorrow:** Lifetime elision rules! 📋